# Training Data

## Task 1: Create Azure Blob container 'librispeech-raw' 

## Task 2: Download  Dataset -> Colab -> Blob Storage 


In [ ]:
!pip install -q azure-storage-blob azure-identity azure-keyvault-secrets python-dotenv tqdm

import shutil, subprocess

if shutil.which("azcopy") is None:
    print("azcopy not found — installing...")
    !wget -q https://aka.ms/downloadazcopy-v10-linux -O /tmp/azcopy.tar.gz
    !tar -xf /tmp/azcopy.tar.gz -C /tmp
    !cp /tmp/azcopy_linux_amd64_*/azcopy /usr/local/bin/azcopy
    !chmod +x /usr/local/bin/azcopy

print("azcopy:", shutil.which("azcopy"))
print(subprocess.run(["azcopy", "--version"], capture_output=True, text=True).stdout.strip())


In [ ]:
import os
import shutil
import subprocess
import tarfile
import time
import urllib.request
from pathlib import Path
from datetime import datetime, timedelta, timezone

from azure.storage.blob import (
    BlobServiceClient,
    generate_container_sas,
    ContainerSasPermissions,
)


class LibriSpeechPipeline:
    """Download LibriSpeech splits, extract, push to Azure Blob, verify, clean up."""

    SPLITS = {
        "train-clean-100": "https://www.openslr.org/resources/12/train-clean-100.tar.gz",
        "train-clean-360": "https://www.openslr.org/resources/12/train-clean-360.tar.gz",
        "train-other-500": "https://www.openslr.org/resources/12/train-other-500.tar.gz",
        "dev-clean":       "https://www.openslr.org/resources/12/dev-clean.tar.gz",
        "test-clean":      "https://www.openslr.org/resources/12/test-clean.tar.gz",
    }

    def __init__(self, work_dir: str = "/content/librispeech"):
        self.work_dir = Path(work_dir)
        self.work_dir.mkdir(parents=True, exist_ok=True)

    def _log(self, split: str, msg: str) -> None:
        ts = time.strftime("%H:%M:%S")
        print(f"[{ts}] [{split}] {msg}", flush=True)

    def _tar_path(self, split: str) -> Path:
        return self.work_dir / f"{split}.tar.gz"

    def _extract_root(self, split: str) -> Path:
        return self.work_dir / split

    def _extracted_split_dir(self, split: str) -> Path:
        return self._extract_root(split) / "LibriSpeech" / split

    def _assert_split(self, split: str) -> None:
        if split not in self.SPLITS:
            raise ValueError(f"unknown split '{split}'. valid: {list(self.SPLITS)}")

    def _dir_size(self, path: Path) -> int:
        return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())

    def download(self, split: str) -> Path:
        self._assert_split(split)
        url = self.SPLITS[split]
        dest = self._tar_path(split)

        if dest.exists() and dest.stat().st_size > 0:
            self._log(split, f"tarball already present at {dest} ({dest.stat().st_size / 1e9:.2f} GB) — skipping download")
            return dest

        for attempt in range(1, 4):
            try:
                self._log(split, f"downloading {url} (attempt {attempt}/3)")
                tmp = dest.with_suffix(dest.suffix + ".part")
                start = time.time()

                with urllib.request.urlopen(url, timeout=60) as resp, open(tmp, "wb") as f:
                    total = int(resp.headers.get("Content-Length", 0))
                    read = 0
                    chunk = 1 << 20
                    last_report = 0
                    while True:
                        buf = resp.read(chunk)
                        if not buf:
                            break
                        f.write(buf)
                        read += len(buf)
                        if total and read - last_report > 250 * chunk:
                            pct = 100.0 * read / total
                            self._log(split, f"  ...{read / 1e9:.2f} / {total / 1e9:.2f} GB ({pct:.1f}%)")
                            last_report = read
                    if total:
                        self._log(split, f"  ...{read / 1e9:.2f} / {total / 1e9:.2f} GB (100.0%)")

                tmp.rename(dest)
                elapsed = time.time() - start
                self._log(split, f"downloaded {dest.stat().st_size / 1e9:.2f} GB in {elapsed:.0f}s")
                return dest
            except Exception as e:
                self._log(split, f"download attempt {attempt} failed: {e}")
                if attempt == 3:
                    raise
                self._log(split, "waiting 30s before retry")
                time.sleep(30)

    def extract(self, split: str) -> Path:
        self._assert_split(split)
        tar = self._tar_path(split)
        if not tar.exists():
            raise FileNotFoundError(f"tarball missing for {split}: {tar}")

        out_root = self._extract_root(split)
        extracted_dir = self._extracted_split_dir(split)
        if extracted_dir.exists() and any(extracted_dir.iterdir()):
            self._log(split, f"already extracted at {extracted_dir} — skipping")
            return extracted_dir

        out_root.mkdir(parents=True, exist_ok=True)
        self._log(split, f"extracting {tar} -> {out_root}")
        start = time.time()
        with tarfile.open(tar, "r:gz") as tf:
            tf.extractall(out_root, filter="data")
        self._log(split, f"extracted in {time.time() - start:.0f}s")
        return extracted_dir

    def upload_to_azure(self, split: str, connection_string: str, container_name: str) -> None:
        self._assert_split(split)
        src = self._extracted_split_dir(split)
        if not src.exists():
            raise FileNotFoundError(f"extracted folder missing for {split}: {src}")

        for attempt in range(1, 4):
            try:
                svc = BlobServiceClient.from_connection_string(connection_string)
                account_key = svc.credential.account_key
                sas = generate_container_sas(
                    account_name=svc.account_name,
                    container_name=container_name,
                    account_key=account_key,
                    permission=ContainerSasPermissions(read=True, write=True, list=True, create=True, add=True),
                    expiry=datetime.now(timezone.utc) + timedelta(hours=12),
                )
                dest_url = f"https://{svc.account_name}.blob.core.windows.net/{container_name}/{split}?{sas}"

                self._log(split, f"uploading {src} -> blob container '{container_name}'/{split} (attempt {attempt}/3)")
                start = time.time()
                result = subprocess.run(
                    ["azcopy", "copy", f"{src}/*", dest_url, "--recursive=true", "--overwrite=ifSourceNewer"],
                    capture_output=True,
                    text=True,
                )
                if result.returncode != 0:
                    self._log(split, "azcopy FAILED")
                    print(result.stdout)
                    print(result.stderr)
                    raise RuntimeError(f"azcopy upload failed for {split}")
                self._log(split, f"upload complete in {time.time() - start:.0f}s")
                return
            except Exception as e:
                self._log(split, f"upload attempt {attempt} failed: {e}")
                if attempt == 3:
                    raise
                self._log(split, "waiting 30s before retry")
                time.sleep(30)

    def verify(self, split: str, connection_string: str, container_name: str) -> bool:
        self._assert_split(split)
        src = self._extracted_split_dir(split)
        if not src.exists():
            raise FileNotFoundError(f"extracted folder missing for {split}: {src}")

        local_files = [p for p in src.rglob("*") if p.is_file()]
        local_count = len(local_files)
        local_bytes = sum(p.stat().st_size for p in local_files)
        self._log(split, f"local file count: {local_count}, total bytes: {local_bytes}")

        svc = BlobServiceClient.from_connection_string(connection_string)
        container = svc.get_container_client(container_name)
        prefix = f"{split}/"
        remote_count = 0
        remote_bytes = 0
        for blob in container.list_blobs(name_starts_with=prefix):
            remote_count += 1
            remote_bytes += blob.size
        self._log(split, f"remote file count under '{prefix}': {remote_count}, total bytes: {remote_bytes}")

        ok = local_count == remote_count and local_bytes == remote_bytes
        if ok:
            self._log(split, "verify OK ✔")
        else:
            self._log(split, f"verify MISMATCH ✘ (local={local_count}/{local_bytes}B, remote={remote_count}/{remote_bytes}B)")
        return ok

    def cleanup(self, split: str) -> None:
        self._assert_split(split)
        tar = self._tar_path(split)
        extracted = self._extract_root(split)
        freed = 0

        if tar.exists():
            freed += tar.stat().st_size
        if extracted.exists():
            freed += self._dir_size(extracted)

        if tar.exists():
            tar.unlink()
            self._log(split, f"deleted {tar}")
        if extracted.exists():
            shutil.rmtree(extracted, ignore_errors=True)
            self._log(split, f"deleted {extracted}")

        self._log(split, f"freed ~{freed / 1e9:.2f} GB")

    def run(self, split: str, connection_string: str, container_name: str) -> None:
        self._assert_split(split)
        self._log(split, "=== START ===")
        self.download(split)
        self.extract(split)
        self.upload_to_azure(split, connection_string, container_name)
        ok = self.verify(split, connection_string, container_name)
        if not ok:
            self._log(split, "skipping cleanup because verify failed")
            return
        self.cleanup(split)
        self._log(split, "=== DONE ===")


In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import ClientSecretCredential
from azure.keyvault.secrets import SecretClient

load_dotenv()

client_id     = os.environ["AZURE_CLIENT_ID"]
tenant_id     = os.environ["AZURE_TENANT_ID"]
client_secret = os.environ["AZURE_CLIENT_SECRET"]
vault_url     = os.environ["AZURE_KEY_VAULT_URL"]

credential = ClientSecretCredential(
    tenant_id=tenant_id,
    client_id=client_id,
    client_secret=client_secret,
)
secret_client = SecretClient(vault_url=vault_url, credential=credential)
connection_string = secret_client.get_secret("AZURE-STORAGE-CONNECTION-STRING").value

container_name = "librispeech-raw"

pipeline = LibriSpeechPipeline()

splits = ['train-clean-100', 'train-clean-360', 'train-other-500', 'dev-clean', 'test-clean']

for split in splits:
    pipeline.run(split, connection_string, container_name)
